# LLM · Lecture 10 — Agents & Tool Calling, live on the class model

An LLM can only emit text — it can't do arithmetic reliably, look up today's weather, or query your database. **Tool calling** fixes that: you describe a function to the model, it asks you to call it, *you* run it, feed the result back, and it continues. Wrap that round-trip in a loop and you have an **agent**.

This notebook runs it for real against the **class model** (`gemma-4-E4B-it` behind the course LiteLLM proxy), which returns **native `tool_calls`** — the same shape as the OpenAI API, so what you build here transfers straight to production. You'll:

1. see *why* a tool is needed (the model fails a calculation on its own),
2. define a **tool** = a function + a JSON-schema description,
3. trace **one round-trip** and watch the `messages` list grow,
4. watch a **ReAct** step reason out loud,
5. build a **tiny agent loop** (dispatch, `max_steps`, duplicate guard, parallel calls),
6. see it **recover from a bad tool call**.

Uses the standard `openai` client (preinstalled in Colab). The pure-Python parts (tools, dispatch, the agent function) always run; the **live model calls are key-guarded**, so paste your course API key to see them fire.

## 0. Setup — point the OpenAI client at the class proxy

In [ ]:
import os, json

os.environ.setdefault('OPENAI_BASE_URL', 'https://llm.nat-d.uk/v1')   # the class proxy
# os.environ['OPENAI_API_KEY'] = 'sk-...'   # <-- paste your key (portal: Profile -> API key)
MODEL = 'gemma-4-E4B-it'
HAVE_KEY = bool(os.environ.get('OPENAI_API_KEY'))

def client():
    from openai import OpenAI          # preinstalled in Colab
    return OpenAI()                    # reads OPENAI_API_KEY + OPENAI_BASE_URL

def need_key():
    print('No OPENAI_API_KEY set — set it in the cell above to run the live calls.')
    print('(Get one from the course portal: Profile -> API key.)')
print('base:', os.environ['OPENAI_BASE_URL'], '| model:', MODEL, '| key set:', HAVE_KEY)

## 1. Why bother? Watch the model fail without a tool

Language models are not calculators — they predict tokens. Ask for an exact product of two biggish numbers and a small model often gets it *slightly* wrong. That miss is the whole motivation for tools.

In [ ]:
question = 'What is 6473 * 8219? Reply with only the number.'
if HAVE_KEY:
    got = client().chat.completions.create(
        model=MODEL, messages=[{'role':'user','content':question}], temperature=0
    ).choices[0].message.content
    print('model:', got.strip())
    print('truth:', 6473 * 8219, '  <- the model often misses the last digits')
else:
    need_key()

## 2. What is a tool? A function + a JSON-schema description

A tool is two things: a plain Python function, and a JSON-schema **description** you hand the model so it knows the tool exists and what arguments it takes. The model can't run your code — the description is how it learns to *ask*. **Never `eval()` raw model output** — guard the input.

In [ ]:
def calculator(expression):
    # only digits and arithmetic — never eval() arbitrary model output blindly
    if not set(expression) <= set('0123456789+-*/(). '):
        raise ValueError('only numbers and + - * / ( ) allowed')
    return str(eval(expression))

calculator_spec = {
    'type': 'function',
    'function': {
        'name': 'calculator',
        'description': "Evaluate a math expression like '12 * 8'.",
        'parameters': {
            'type': 'object',
            'properties': {'expression': {'type': 'string'}},
            'required': ['expression'],
        },
    },
}
print(calculator('4831 * 1279'), '  <- exact, because Python did the math')

## 3. One round-trip, traced step by step

The cycle: call the model with `messages` **and** `tools=[...]`; it replies with `tool_calls` (not text); **you** run the function and append a `role:"tool"` message tagged by the call id; call the model again — it reads the result and writes the answer. Two easy-to-miss details: `arguments` is a **JSON string** (`json.loads` it), and the `tool` message must carry the call's `id`. Watch the `messages` list grow from 1 to 4.

In [ ]:
def show(messages, label):
    print(f'--- messages after {label}: {len(messages)} ---')
    for m in messages:
        if isinstance(m, dict):                       # our user/tool messages
            print(f'  {m["role"]:9} {m.get("content", "")!r}')
        elif m.tool_calls:                            # assistant requesting a tool
            print(f'  {m.role:9} tool_calls={[c.function.name for c in m.tool_calls]}')
        else:                                         # assistant final answer
            print(f'  {m.role:9} {m.content!r}')

tool_specs = [calculator_spec]

if HAVE_KEY:
    c = client()
    messages = [{'role':'user','content':'A box holds 12 rows of 8 cookies. How many total?'}]
    show(messages, 'user question')
    # turn 1: model requests the tool (no content, just a tool_call). temperature=0 -> deterministic.
    msg = c.chat.completions.create(model=MODEL, messages=messages, tools=tool_specs,
                                    temperature=0).choices[0].message
    messages.append(msg)
    show(messages, 'assistant tool request')
    # YOU run it; append the result as a role:'tool' message tagged by id
    call = msg.tool_calls[0]
    result = calculator(**json.loads(call.function.arguments))   # arguments is a JSON string
    messages.append({'role':'tool','tool_call_id':call.id,'content':result})
    show(messages, 'tool result')
    # turn 2: model reads the result, writes the final answer (no tool call)
    final = c.chat.completions.create(model=MODEL, messages=messages, tools=tool_specs,
                                      temperature=0).choices[0].message
    messages.append(final)
    show(messages, 'final answer')
else:
    need_key()

Read the final list top to bottom: **user** asks → **assistant** requests a tool → **tool** carries the result back → **assistant** answers. Every later step sees the *whole* history, which is why the model can use a `96` it never computed. The tool-request message has **no text content** — the request *is* the message.

## 4. Seeing the *thought* — a ReAct step (prompted-JSON)

Native `tool_calls` carry no prose, so you can't see *why* the model acted. **ReAct** (Reason+Act) makes the thought explicit: drop to the prompted-JSON convention and ask the model to emit a `thought` beside its action. (This "reply with ONLY JSON" prompt *suppresses* the native path — it's an either/or, per the lecture.) Parse defensively; feed observations back.

In [ ]:
SYSTEM = """You solve tasks step by step. You have one tool: calculator(expression).
Reply with ONLY a JSON object, nothing else.
To use the tool:  {"thought": "<why>", "tool": "calculator", "args": {"expression": "..."}}
When done:        {"thought": "<why>", "answer": "<final answer>"}"""

if HAVE_KEY:
    c = client()
    messages = [{'role':'system','content':SYSTEM},
                {'role':'user','content':'What is (18 * 7) + 100?'}]
    for step in range(1, 5):
        raw = c.chat.completions.create(model=MODEL, messages=messages,
                                        temperature=0).choices[0].message.content
        try:
            action = json.loads(raw)          # untrusted: parse defensively
        except json.JSONDecodeError:
            print('FINAL (prose):', raw); break
        print(f'STEP {step}  THOUGHT  {action.get("thought")!r}')
        if 'answer' in action:
            print('FINAL:', action['answer']); break
        try:                                  # a tool error is an observation, not a crash
            result = calculator(**action['args'])
        except Exception as e:
            result = f'error: {type(e).__name__}: {e}'
        print(f'          ACT  calculator({action["args"]})  OBS  {result}')
        messages.append({'role':'assistant','content':raw})
        messages.append({'role':'user','content':f'observation: {result}'})
else:
    need_key()

The visible `thought` is the point of ReAct — it keeps the model honest about *why* it acts and makes the trace easy to debug. You ship the **native** path (next); you reach for ReAct-with-prose when you need to *see it think*.

## 5. A tiny agent — the loop, with dispatch + guardrails

An agent is the round-trip wrapped in a loop: **observe → think → act → repeat**, stopping when the model replies with no tool call. Two real tools now. The key robustness trick: `dispatch()` **always returns a string** — a bad call becomes an error the model can *read and recover from*, never a crash. Guardrails baked in: a hard `max_steps`, and a duplicate-call guard so it can't spin on the same action.

In [ ]:
def lookup(city):
    fake_db = {'bangkok':'33C, humid', 'tokyo':'18C, clear', 'paris':'12C, rain'}
    return fake_db.get(city.strip().lower(), 'no data')

TOOLS = {'calculator': calculator, 'lookup': lookup}
tool_specs = [
    calculator_spec,
    {'type':'function','function':{'name':'lookup',
        'description':'Get the weather for a city name.',
        'parameters':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}},
]

def dispatch(name, raw_args):
    # ALWAYS return a string: every failure becomes an error the model can recover from.
    if name not in TOOLS:
        return f'error: unknown tool {name!r}'
    try:
        args = json.loads(raw_args or '{}')            # arguments is a JSON string
    except json.JSONDecodeError:
        return f'error: arguments were not valid JSON: {raw_args!r}'
    try:
        return str(TOOLS[name](**args))
    except Exception as e:                              # bad args / tool raised
        return f'error: {type(e).__name__}: {e}'

def agent(question, max_steps=5, verbose=True):
    c = client()
    messages = [{'role':'user','content':question}]
    seen = set()
    for step in range(1, max_steps + 1):               # guardrail: bounded steps
        msg = c.chat.completions.create(model=MODEL, messages=messages,
                                        tools=tool_specs, temperature=0).choices[0].message
        if not getattr(msg, 'tool_calls', None):       # STOP: no tool call = final answer
            return msg.content or ''
        messages.append(msg)                           # the assistant's request(s)
        for call in msg.tool_calls:                    # may be several at once (parallel)
            sig = (call.function.name, call.function.arguments)
            if verbose: print(f'STEP {step}  ACT  {call.function.name}({call.function.arguments})')
            if sig in seen:                            # no-progress / duplicate guard
                result = 'error: already called with these args; try something else'
            else:
                seen.add(sig)
                result = dispatch(call.function.name, call.function.arguments)
            if verbose: print(f'          OBS  {result}')
            messages.append({'role':'tool','tool_call_id':call.id,'content':result})
    return 'stopped: hit max_steps'                     # termination guardrail

if HAVE_KEY:
    print('FINAL:', agent("What's 18 * 7, and what's the weather in Tokyo?"))
else:
    need_key()

The class model emits **both** tool calls in one assistant message (a *parallel* call), so you append **two** `tool` messages, then call once more and it answers — ending the loop. At `temperature=0` the trace is identical every run (reproducible dispatch, fewer hallucinated tool names).

## 6. Recovering from a bad tool call

`dispatch()` earns its keep when a call goes wrong. Force a bad first call (the model tries to evaluate worded text), feed the **error string** back, and watch the agent read it and retry with a valid expression — because a tool failure is just one more observation, not a crash.

In [ ]:
if HAVE_KEY:
    c = client()
    bad_args = '{"expression": "one hundred forty-four / twelve"}'
    messages = [
        {'role':'user','content':'What is 144 divided by 12?'},
        {'role':'assistant','tool_calls':[{'id':'bad1','type':'function',
            'function':{'name':'calculator','arguments':bad_args}}]},
    ]
    err = dispatch('calculator', bad_args)
    print('OBS (bad call):', err)
    messages.append({'role':'tool','tool_call_id':'bad1','content':err})
    for step in range(2, 6):
        msg = c.chat.completions.create(model=MODEL, messages=messages,
                                        tools=tool_specs, temperature=0).choices[0].message
        if not msg.tool_calls:
            print('FINAL:', msg.content); break
        messages.append(msg)
        for call in msg.tool_calls:
            print(f'STEP {step}  ACT  {call.function.name}({call.function.arguments})')
            result = dispatch(call.function.name, call.function.arguments)
            print(f'          OBS  {result}')
            messages.append({'role':'tool','tool_call_id':call.id,'content':result})
else:
    need_key()

The error went back into the history; the model self-corrected to `144 / 12`. **Return errors, don't raise** — a raised exception kills the loop and gives the model no chance to recover.

## Your turn

1. **Add a tool.** Write `word_count(text)` + its schema, register it in `TOOLS`/`tool_specs`, and ask the agent "how many words are in this sentence: …". Does it pick the right tool?
2. **Break the guardrail.** Give the agent a task it *can't* finish with these tools (e.g. "what's the capital of France?" with only calculator/lookup) and watch `max_steps` stop it.
3. **Duplicate guard.** Force the same tool call twice (same args) and confirm the second gets the "already called" error instead of re-running.
4. **Temperature.** Re-run §5 at `temperature=0.8`. Does tool choice stay stable? Why is `temperature=0` the right default for dispatch?
5. **Native vs prompted.** The §4 ReAct loop feeds observations back as `user` messages; the §5 agent uses `role:"tool"` messages. Explain the message-accounting difference.

## Recap
- A **tool** = a function + a JSON-schema description; the model *asks*, your code *runs* it.
- The **round-trip**: `messages`+`tools` → `tool_calls` → you run it → append a `role:"tool"` result (by id) → call again. `arguments` is a JSON **string**.
- An **agent** loops that until the model answers with no tool call; **`dispatch()` returns a string** so failures are recoverable; guard with **`max_steps`** + a **duplicate check** + **`temperature=0`**.
- The class model returns **native `tool_calls`** (OpenAI-shaped) — production-transferable; the **prompted-JSON** convention is the fallback for models with no tool support (and lets you *see the thought*).